# Institution type and results: independent, academy converter, sponsor-led, free school, maintained, college

The England-wide model so far knows where an institution is (region, local authority) and what its GCSE profile is, but not **what kind of institution it is**. This notebook adds the public register's classification and asks how type relates to results, at GCSE and at A-level, keeping to the seven high-level subject groups (Maths, Sciences, English, Humanities, Social sciences, Business & Computing, Creative arts).

| Category | What it is | Institutions (with GCSE results / A-level only) |
| --- | --- | --- |
| **Independent (fee-paying)** | Independent schools | 0 / 576 |
| **Academy converter** | Schools that chose to convert to academy status (mostly previously rated good or outstanding) | 1,079 / 21 |
| **Academy sponsor-led** | Academies created to replace or take over a school that was under-performing, run with a sponsor | 343 / 5 |
| **Free school / UTC / studio** | New state-funded schools set up outside the local authority (a third route into the academy sector) | 139 / 3 |
| **LA maintained** | Community, voluntary aided, voluntary controlled and foundation schools | 303 / 1 |
| **College (post-16)** | Further-education colleges, sixth-form centres and 16-19 academies and free schools | 9 / 253 |
| **Other** | Special schools and other provision | 10 / 5 |

The separation of the two academy routes is the point of the notebook: they were **created for different reasons and from different starting points**, so a difference between them at any date is not the same as an effect of conversion.

**How type enters the model**

- **GCSE** (for the four state types that have GCSE results): a type shift in general GCSE quality, in the tilt and in consistency, on top of region and local-authority effects.
- **A-level, beyond GCSE:** for each subject group, a type shift that remains after the school's GCSE profile is allowed for. Independent schools, colleges and other institutions have no GCSE results, so for them the type shift is an offset that also contains whatever their GCSE profile would have been.
- **Time as an academy:** for converters and sponsor-led academies, how general GCSE quality and the shared A-level quality change per decade since the school became an academy (register opening date).

**What this can and cannot tell us.** These are one year of results, so everything here is association. The two academy groups differ in intake and history before they were academies, and converters were selected for being strong; a gap between them mostly reflects that. Years-as-an-academy is also confounded with which schools converted when. Nothing here measures what would have happened to a school had it not converted.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
import xarray as xr

%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")
print(f"Running on PyMC v{pm.__version__}")

## Data

In [ ]:
raw = pd.read_csv("data/all-value-add-errors.csv")
names = pd.read_csv("data/school-names.csv").set_index("URN")

# ---- GCSE: six elements per school with known standard errors (schools with Progress 8 results)
elements = ["English", "Maths", "Science", "Humanities", "Languages", "Open"]
column = {"English": "P8MEAENG", "Maths": "P8MEAMAT", "Science": "SCIVAMEA_PTQ_EE",
          "Humanities": "HUMVAMEA_PTQ_EE", "Languages": "LANVAMEA_PTQ_EE", "Open": "P8MEAOPEN"}
frames = []
for e in elements:
    c = column[e]
    sub = raw[["URN", c, f"{c} lower", f"{c} upper"]].dropna()
    sub.columns = ["URN", "va", "lower", "upper"]
    sub["element"] = e
    frames.append(sub)
long = pd.concat(frames)
long["se"] = (long["upper"] - long["lower"]) / (2 * 1.96)
n_el = long.groupby("URN")["element"].nunique()
long = long[long["URN"].isin(n_el[n_el >= 3].index)].sort_values("URN").reset_index(drop=True)

# ---- all institutions: schools with GCSE results first, then institutions with A-level results only
urns_gcse = pd.Index(sorted(long["URN"].unique()))
urns_only = pd.Index(sorted(set(raw["URN"]) - set(urns_gcse)))
inst = urns_gcse.append(urns_only)
n_g, n_inst = len(urns_gcse), len(inst)
no_gcse = (np.arange(n_inst) >= n_g).astype(float)

long["school_idx"] = urns_gcse.get_indexer(long["URN"])
long["element_idx"] = long["element"].map({e: k for k, e in enumerate(elements)}).to_numpy()
n_elements = len(elements)
x_obs, x_se = long["va"].to_numpy(), long["se"].to_numpy()
s_idx, e_idx = long["school_idx"].to_numpy(), long["element_idx"].to_numpy()

region_series = raw.drop_duplicates("URN").set_index("URN")["RGN24NM"].reindex(inst)
regions = list(region_series.value_counts().index)
reg_idx = region_series.map({r: k for k, r in enumerate(regions)}).fillna(len(regions)).astype(int).to_numpy()   # unknown region -> national average
is_london = (region_series.to_numpy() == "London")
print(f"{n_g} schools with GCSE results, {len(urns_only)} institutions with A-level results only, {n_inst} in all; {is_london.sum()} in London")


# ---- local authority for every institution
la_series = names["local_authority"].reindex(inst)
la_names = sorted(la_series.unique())
la_idx = la_series.map({a: k for k, a in enumerate(la_names)}).to_numpy()

# ---- institution type, from the public register's detailed establishment type
def type_category(t):
    if t in ("Other independent school", "Other independent special school"):
        return "independent (fee-paying)"
    if t in ("Academy converter", "Academy special converter"):
        return "academy converter"
    if t in ("Academy sponsor led", "Academy special sponsor led"):
        return "academy sponsor-led"
    if t in ("Free schools", "Free schools special", "University technical college", "Studio schools", "City technology college", "Free schools alternative provision"):
        return "free school / UTC / studio"
    if t in ("Community school", "Voluntary aided school", "Voluntary controlled school", "Foundation school"):
        return "LA maintained"
    if t in ("Further education", "Sixth form centres", "Academy 16-19 converter", "Academy 16 to 19 sponsor led", "Free schools 16 to 19"):
        return "college (post-16)"
    return "other"
type_names = ["independent (fee-paying)", "academy converter", "academy sponsor-led", "free school / UTC / studio", "LA maintained", "college (post-16)", "other"]
state4 = type_names[1:5]                      # state-funded types that have GCSE results
other3 = [type_names[0], type_names[5], type_names[6]]
inst_type = np.array([type_names.index(type_category(t)) for t in names["type_detail"].reindex(inst).to_numpy()])
is4 = ((inst_type >= 1) & (inst_type <= 4)).astype(float)
t4_idx = np.where(is4 == 1, inst_type - 1, 0)
is3 = 1.0 - is4
t3_idx = np.array([{0: 0, 5: 1, 6: 2}.get(k, 0) for k in inst_type])

# years as an academy (converters and sponsor-led), in decades, centred on a typical 12 years
open_year = pd.to_datetime(names["open_date"].reindex(inst), format="%d-%m-%Y", errors="coerce").dt.year.to_numpy()
is_cs = ((inst_type == 1) | (inst_type == 2)).astype(float)
cs_idx = np.where(inst_type == 2, 1, 0)      # 0 converter, 1 sponsor-led
years_c = is_cs * ((2024 - np.nan_to_num(open_year, nan=2012)) / 10.0 - 1.2)
summary = pd.crosstab(pd.Series(np.array(type_names)[inst_type], name="type"), pd.Series(np.where(no_gcse == 1, "A-level results only", "has GCSE results"), name=""))
print(summary.reindex(type_names).fillna(0).astype(int).to_string())

In [ ]:
arts = ["Art & Design", "Art & Design (Fine Art)", "Art & Design (Photography)", "Art & Design (Graphics)", "Art & Design (Textiles)",
        "Art & Design (3d Studies)", "Art & Design (Critical Studies)", "Music", "Music Technology", "Drama & Theatre Studies", "Dance"]
groups = {"Maths": ["Mathematics"],
          "Sciences": ["Biology", "Chemistry", "Physics"],
          "English": ["English Literature", "English Language", "English Language & Literature"],
          "Humanities": ["History", "Geography", "Religious Studies", "Logic/ Philosophy", "Ancient History", "Classical Civilisation"],
          "Social sciences": ["Psychology", "Sociology", "Economics", "Government & Politics", "Law"],
          "Business & Computing": ["Business Studies:Single", "Computer Studies/Computing"],
          "Creative arts": arts}
group_names = list(groups)
n_groups = len(group_names)

raw_i = raw.set_index("URN").reindex(inst)
frames = []
for gname, subjects in groups.items():
    va = pd.DataFrame({s: raw_i[f"A-level {s} VA"] for s in subjects})
    se = pd.DataFrame({s: (raw_i[f"A-level {s} VA upper"] - raw_i[f"A-level {s} VA lower"]) / (2 * 1.96) for s in subjects})
    ent = pd.DataFrame({s: raw_i[f"A-level {s} entries"].where(va[s].notna(), 0).fillna(0) for s in subjects})
    total = ent.sum(axis=1)
    pooled_va = (va.fillna(0) * ent).sum(axis=1) / total.replace(0, np.nan)
    pooled_se = np.sqrt(((se.fillna(0) * ent) ** 2).sum(axis=1)) / total.replace(0, np.nan)   # entry-weighted; assumes separate cohorts
    f = pd.DataFrame({"inst_idx": np.arange(n_inst), "group": gname, "va": pooled_va.to_numpy(), "se": pooled_se.to_numpy(), "entries": total.to_numpy()})
    frames.append(f.dropna(subset=["va", "se"]))
alevel = pd.concat(frames).reset_index(drop=True)
alevel["group_idx"] = alevel["group"].map({g: k for k, g in enumerate(group_names)}).to_numpy()
y_obs, y_se = alevel["va"].to_numpy(), alevel["se"].to_numpy()
ys_idx, yg_idx = alevel["inst_idx"].to_numpy(), alevel["group_idx"].to_numpy()
assert (y_se > 0).all()
print(f"{len(alevel)} institution-group A-level observations")

### First look: raw averages by type

Average value added by institution type, before any modelling: a GCSE composite (the mean of a school's English, Maths, Science, Humanities and Open scores) and each A-level subject group, with the number of institutions behind each cell. Cells with fewer than 20 institutions are shown in grey.

In [ ]:
gcse_comp = long[long["element"].isin(["English", "Maths", "Science", "Humanities", "Open"])].groupby("school_idx")["va"].mean()
rows = {}
for k, tname in enumerate(type_names):
    r = {}
    idx = np.where(inst_type[:n_g] == k)[0]
    v = gcse_comp.reindex(idx).dropna()
    r["GCSE composite"] = (v.mean() if len(v) else np.nan, len(v))
    for gname in group_names:
        d = alevel[(alevel["group"] == gname) & (inst_type[alevel["inst_idx"].to_numpy()] == k)]
        r[gname] = (d["va"].mean() if len(d) else np.nan, len(d))
    rows[tname] = r
cols = ["GCSE composite"] + group_names
means = pd.DataFrame({c: {t: rows[t][c][0] for t in type_names} for c in cols})
ns_ = pd.DataFrame({c: {t: rows[t][c][1] for t in type_names} for c in cols})
display(means.round(3))
fig, ax = plt.subplots(figsize=(11, 4.2))
im = ax.imshow(means.to_numpy(), cmap="RdBu_r", vmin=-0.6, vmax=0.6, aspect="auto")
for i in range(means.shape[0]):
    for j in range(means.shape[1]):
        v, n = means.iloc[i, j], ns_.iloc[i, j]
        if n > 0:
            ax.text(j, i, f"{v:.2f}\nn={n}", ha="center", va="center", fontsize=7.5, color="grey" if n < 20 else "black")
ax.set_xticks(range(len(cols)), cols, rotation=30, ha="right"); ax.set_yticks(range(len(type_names)), type_names)
ax.grid(False); ax.set_title("Raw mean value added by institution type")
plt.tight_layout(); plt.show()

## Model

In [ ]:
keep = np.array([0.0 if e == "Humanities" else 1.0 for e in elements])

def build_model(interact=False):
    """interact=True adds London-specific type effects (extra shifts for London institutions, by type)."""
    coords = {"element": elements, "region": regions, "group": group_names, "la": la_names, "type4": state4, "type3": other3,
              "cs": ["academy converter", "academy sponsor-led"]}
    with pm.Model(coords=coords) as model:
        # ---- geography and institution type: where general GCSE quality sits
        sigma_m = pm.HalfNormal("sigma_m", 0.5)
        m = pm.ZeroSumNormal("m", sigma=sigma_m, dims="region")
        m_all = pt.concatenate([m, pt.zeros(1)])
        sigma_a = pm.HalfNormal("sigma_a", 0.3)
        a_la = pm.Normal("a_la", 0, sigma_a, dims="la")
        type_g = pm.ZeroSumNormal("type_g", sigma=0.5, dims="type4")       # type shift in general GCSE quality (state types with GCSE results)
        type_h = pm.ZeroSumNormal("type_h", sigma=0.5, dims="type4")       # type shift in the tilt
        type_c = pm.ZeroSumNormal("type_c", sigma=0.3, dims="type4")       # type shift in log consistency
        slope_g = pm.Normal("slope_g", 0, 0.3, dims="cs")                  # change in general quality per decade as an academy
        g_loc = m_all[reg_idx] + a_la[la_idx] + is4 * type_g[t4_idx] + is_cs * years_c * slope_g[cs_idx]
        if interact:
            ix_g = pm.ZeroSumNormal("ix_g", sigma=0.3, dims="type4")            # extra type shift in general GCSE quality for London
            g_loc = g_loc + is_london.astype(float) * is4 * ix_g[t4_idx]
        h_loc = is4 * type_h[t4_idx]

        # ---- GCSE side, schools with GCSE results only
        mu = pm.Normal("mu", 0, 1, dims="element")
        lam = pm.HalfNormal("lam", 1, dims="element")
        tau = pm.HalfNormal("tau", 0.5, dims="element")
        g = pm.Normal("g", g_loc[:n_g], 1, shape=n_g)
        h = pm.Normal("h", h_loc[:n_g], 1, shape=n_g)
        k_raw = pm.Normal("kappa_raw", 0, 0.5, shape=n_elements)
        kappa = pm.Deterministic("kappa", k_raw * keep, dims="element")   # Humanities fixed at 0; the sign of the tilt is fixed after sampling
        sigma_s = pm.HalfNormal("sigma_s", 0.5)
        rho = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
        w = pm.Normal("w", 0, 1, shape=n_g)
        log_s = is4[:n_g] * type_c[t4_idx[:n_g]] + sigma_s * (rho * (g - g_loc[:n_g]) + pt.sqrt(1 - rho**2) * w)   # not stored: unused below
        pm.Normal("x_obs", mu=mu[e_idx] + lam[e_idx] * g[s_idx] + kappa[e_idx] * h[s_idx],
                  sigma=pt.sqrt((tau[e_idx] * pt.exp(log_s[s_idx]))**2 + x_se**2), observed=x_obs)

        # institutions without GCSE results: the typical GCSE quality for where they are and what they are
        g_full = pt.concatenate([g, g_loc[n_g:]])
        h_full = pt.concatenate([h, h_loc[n_g:]])

        # ---- A-level side (all institutions)
        nu = pm.Normal("nu", 0, 0.5, dims="group")
        sd_group = pm.HalfNormal("sd_group", 0.5, dims="group")
        sd_group_ng = pm.HalfNormal("sd_group_ng", 0.5, dims="group")
        sigma_psi = pm.HalfNormal("sigma_psi", 0.3)
        psi = pm.ZeroSumNormal("psi", sigma=sigma_psi, dims="region")
        psi_all = pt.concatenate([psi, pt.zeros(1)])
        sigma_xi = pm.HalfNormal("sigma_xi", 0.2)
        xi = pm.Normal("xi", 0, sigma_xi, dims="la")
        slope_u = pm.Normal("slope_u", 0, 0.3, dims="cs")                  # change in shared A-level quality per decade as an academy
        u = pm.Normal("u", 0, 1, shape=n_inst)
        b = pm.Normal("b", 0, 0.5, dims="group")
        c = pm.Normal("c", 0, 0.5, dims="group")
        lam_a = pm.HalfNormal("lam_a", 0.5, dims="group")
        type_a = pm.ZeroSumNormal("type_a", sigma=0.3, dims=("group", "type4"))   # type shift at A-level beyond GCSE, state types, by subject group
        d3 = pm.Normal("d3", 0, 0.5, dims=("group", "type3"))                      # mean shift for independent schools, colleges and others, by group
        ix_term = 0.0
        if interact:
            ix_a = pm.ZeroSumNormal("ix_a", sigma=0.15, dims=("group", "type4"))   # extra type shift at A-level beyond GCSE for London
            ix_term = is_london[ys_idx].astype(float) * is4[ys_idx] * ix_a[yg_idx, t4_idx[ys_idx]]
        shared = psi_all[reg_idx] + xi[la_idx] + is_cs * years_c * slope_u[cs_idx] + u
        y_mean = (nu[yg_idx] + b[yg_idx] * g_full[ys_idx] + c[yg_idx] * h_full[ys_idx] + lam_a[yg_idx] * shared[ys_idx]
                  + is4[ys_idx] * type_a[yg_idx, t4_idx[ys_idx]] + is3[ys_idx] * d3[yg_idx, t3_idx[ys_idx]] + ix_term)
        sd_y = sd_group[yg_idx] * (1 - no_gcse[ys_idx]) + sd_group_ng[yg_idx] * no_gcse[ys_idx]
        pm.Normal("y_obs", mu=y_mean, sigma=pt.sqrt(sd_y**2 + y_se**2), observed=y_obs)
    return model

In [ ]:
model = build_model()

### Fit

Four chains of 500 draws after 1,500 tuning steps. (A shorter run than in the other notebooks: this model stores several thousand per-school quantities per draw, and the run has to fit in the machine's memory. The effective sample sizes below show whether that is enough.)

In [ ]:
with model:
    idata = pm.sample(draws=500, tune=1500, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)

### Diagnostics

In [ ]:
def align_sign(idata, tilt_terms):
    """The tilt is defined only up to sign; flip each chain (in place) to the orientation with Maths and Science positive and English and Open negative."""
    post = idata.posterior
    k = post["kappa"]
    score = (k.sel(element="Maths") + k.sel(element="Science") - k.sel(element="English") - k.sel(element="Open")).mean("draw")
    sign = xr.where(score > 0, 1.0, -1.0)
    for name in tilt_terms:
        post[name] = post[name] * sign
    return post, sign.to_numpy()

post_a, sign_a = align_sign(idata, ["h", "kappa", "c", "type_h"])
n_div = int(idata.sample_stats["diverging"].sum())
print("chains flipped:", int((sign_a < 0).sum()), "of", len(sign_a), "| divergences =", n_div)
post_a = post_a.to_dataset().drop_vars(["w", "kappa_raw", "rho_raw"])      # a plain Dataset; drop what is not needed below
del idata
import gc; gc.collect()

# convergence: every global parameter in full, and every 10th school for the per-school arrays
per_school = ["g", "h", "u"]
glob = [v for v in post_a.data_vars if v not in per_school]
small = xr.Dataset({**{v: post_a[v] for v in glob}, **{v: post_a[v].isel({post_a[v].dims[-1]: slice(0, None, 10)}) for v in per_school}})
rh, es = az.rhat(small), az.ess(small)
worst = pd.DataFrame({"max r_hat": {v: float(rh[v].max()) for v in rh.data_vars}, "min bulk ESS": {v: float(es[v].min()) for v in es.data_vars}}).sort_values("max r_hat", ascending=False)
display(worst.round(3).head(10))
del small, rh, es; gc.collect()

## Regions and local authorities

These are in the model throughout (a shift for each of 9 regions and 152 local authorities, in general GCSE quality and in the shared A-level quality that GCSE does not explain), so every type comparison below is net of where the institutions are. Here they are shown. Left: the regional shift in general GCSE quality (SD units). Right: the regional shift in the A-level quality beyond GCSE (in the scale of the institution-to-institution spread). The table converts both to value-added points at the average loading, and the second table gives the London boroughs.

In [ ]:
def draws(name, thin=1):
    a = post_a[name].to_numpy()
    return a.reshape(-1, *a.shape[2:])[::thin]

def interval(x): return np.percentile(x, [5.5, 50, 94.5], axis=0)
m_d, psi_d = draws("m"), draws("psi")
lam_m, lam_a_m = draws("lam").mean(), draws("lam_a").mean()
order_r = np.argsort(-m_d.mean(axis=0))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, arr, label, col in [(axes[0], m_d, "general GCSE quality (SD units)", "#4C72B0"), (axes[1], psi_d, "A-level shared quality beyond GCSE", "#DD8452")]:
    lo, med, hi = interval(arr)
    for row, k in enumerate(order_r):
        ax.plot([lo[k], hi[k]], [row, row], color=col, linewidth=2.5); ax.plot(med[k], row, "o", color=col)
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--"); ax.set_xlabel(label)
axes[0].set_yticks(range(len(regions)), [regions[k] for k in order_r]); axes[0].invert_yaxis()
plt.tight_layout(); plt.show()
display(pd.DataFrame({"GCSE quality (SD)": m_d.mean(axis=0), "GCSE points": m_d.mean(axis=0) * lam_m, "A-level shift beyond GCSE (points)": psi_d.mean(axis=0) * lam_a_m,
                      "P(A-level shift > 0)": (psi_d > 0).mean(axis=0)}, index=regions).iloc[order_r].round(3))

a_la_d, xi_d = draws("a_la"), draws("xi")
lon_las = [k for k, a in enumerate(la_names) if np.any(is_london & (la_idx == k))]
tab = pd.DataFrame({"schools with GCSE": [int(np.sum(la_idx[:n_g] == k)) for k in lon_las],
                    "GCSE quality shift (SD)": a_la_d[:, lon_las].mean(axis=0),
                    "89% interval": [f"[{np.percentile(a_la_d[:, k], 5.5):+.2f}, {np.percentile(a_la_d[:, k], 94.5):+.2f}]" for k in lon_las],
                    "A-level shared shift": xi_d[:, lon_las].mean(axis=0)}, index=[la_names[k] for k in lon_las]).sort_values("GCSE quality shift (SD)", ascending=False)
display(tab.round(2))

## Type at GCSE

The shift in each of three things for the four state types, relative to the average of the four (they sum to zero), after region and local authority are allowed for: **general quality** (in SD units of the school-to-school spread), the **tilt** (positive is the Maths-and-Science side) and **log consistency** (positive means more scattered across elements). The table gives contrasts between types; "in points" converts general quality into GCSE value-added points at the average element loading.

In [ ]:
def draws(name, thin=1):
    a = post_a[name].to_numpy()
    return a.reshape(-1, *a.shape[2:])[::thin]

def interval(x): return np.percentile(x, [5.5, 50, 94.5], axis=0)
colors4 = ["#4C72B0", "#DD8452", "#55A868", "#8172B3"]
tg, th, tc = draws("type_g"), draws("type_h"), draws("type_c")
lam_mean = draws("lam").mean()

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
for ax, arr, title in [(axes[0], tg, "general GCSE quality (SD units)"), (axes[1], th, "tilt (+ = Maths/Science side)"), (axes[2], tc, "log consistency (+ = more scattered)")]:
    lo, med, hi = interval(arr)
    for k in range(4):
        ax.plot([lo[k], hi[k]], [k, k], color=colors4[k], linewidth=3); ax.plot(med[k], k, "o", color=colors4[k])
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--"); ax.set_xlabel(title)
axes[0].set_yticks(range(4), state4); axes[0].invert_yaxis()
plt.tight_layout(); plt.show()

pairs = [("academy converter", "academy sponsor-led"), ("academy converter", "LA maintained"), ("academy sponsor-led", "LA maintained"), ("free school / UTC / studio", "LA maintained")]
rows = []
for a, b in pairs:
    ia, ib = state4.index(a), state4.index(b)
    for name, arr in [("general quality", tg), ("tilt", th), ("log consistency", tc)]:
        diff = arr[:, ia] - arr[:, ib]
        lo, hi = np.percentile(diff, [5.5, 94.5])
        rows.append({"contrast": f"{a} minus {b}", "quantity": name, "difference": diff.mean(), "89% interval": f"[{lo:.3f}, {hi:.3f}]",
                     "P(> 0)": (diff > 0).mean(), "in GCSE points": diff.mean() * lam_mean if name == "general quality" else np.nan})
pd.DataFrame(rows).set_index(["contrast", "quantity"]).round(3)

## Years as an academy

For converters and sponsor-led academies, the change per decade since opening as an academy (centred on a typical 12 years), in general GCSE quality (SD units; "in points" at the average loading) and in the shared A-level quality that GCSE does not explain (scale of the institution-to-institution spread). A positive slope means academies that have been open **longer** sit higher after allowing for everything else. That is not the same as time as an academy raising results: the academies that opened earliest were different schools, and for converters they were the ones chosen to convert first.

In [ ]:
sg, su = draws("slope_g"), draws("slope_u")
lam_a_mean = draws("lam_a").mean(axis=0)
rows = []
for k, cname in enumerate(["academy converter", "academy sponsor-led"]):
    for name, arr, scale in [("general GCSE quality, per decade", sg, lam_mean), ("shared A-level quality, per decade", su, lam_a_mean.mean())]:
        lo, hi = np.percentile(arr[:, k], [5.5, 94.5])
        rows.append({"type": cname, "slope": name, "mean": arr[:, k].mean(), "89% interval": f"[{lo:.3f}, {hi:.3f}]", "P(> 0)": (arr[:, k] > 0).mean(),
                     "in value-added points": arr[:, k].mean() * scale})
pd.DataFrame(rows).set_index(["type", "slope"]).round(3)

## Type at A-level, beyond GCSE

For each subject group, the shift in A-level value added for each state type **after the school's GCSE profile is allowed for**, relative to the average of the four (value-added points; they sum to zero within a group). A positive value means the type does better at A-level in that subject than its GCSE profile predicts. The second table gives the mean shifts for independent schools, colleges and other institutions; these have no GCSE results, so their shift is measured against a typical GCSE profile for where they are and cannot be separated from what their GCSE profile would have been.

In [ ]:
ta = draws("type_a")            # (n, group, type4)
d3 = draws("d3")                # (n, group, type3)
fig, ax = plt.subplots(figsize=(9, 5.2))
for k in range(4):
    lo, med, hi = np.percentile(ta[:, :, k], [5.5, 50, 94.5], axis=0)
    off = (k - 1.5) * 0.17
    for j in range(n_groups):
        ax.plot([lo[j], hi[j]], [j + off, j + off], color=colors4[k], linewidth=2.2, label=state4[k] if j == 0 else None)
        ax.plot(med[j], j + off, "o", color=colors4[k], markersize=4)
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_yticks(range(n_groups), group_names); ax.invert_yaxis()
ax.set_xlabel("shift in A-level value added beyond GCSE profile (points)"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

rows = []
for a, b in [("academy converter", "academy sponsor-led"), ("academy converter", "LA maintained"), ("academy sponsor-led", "LA maintained")]:
    ia, ib = state4.index(a), state4.index(b)
    r = {"contrast": f"{a} minus {b}"}
    for j, gname in enumerate(group_names):
        diff = ta[:, j, ia] - ta[:, j, ib]
        lo, hi = np.percentile(diff, [5.5, 94.5])
        r[gname] = f"{diff.mean():+.2f} [{lo:+.2f}, {hi:+.2f}]"
    rows.append(r)
display(pd.DataFrame(rows).set_index("contrast"))

rows = []
for k, tname in enumerate(other3):
    r = {"type": tname}
    for j, gname in enumerate(group_names):
        lo, hi = np.percentile(d3[:, j, k], [5.5, 94.5])
        r[gname] = f"{d3[:, j, k].mean():+.2f} [{lo:+.2f}, {hi:+.2f}]"
    rows.append(r)
display(pd.DataFrame(rows).set_index("type"))

## Independent schools alongside the state types

The comparisons above are within the state sector. Here the independent (fee-paying) sector is put beside each state type, subject group by subject group. Both are measured against the same baseline (the average of the four state types, at a typical GCSE profile for where an institution is), so their difference is the gap between an independent school and a state school of that type. Two things to keep in mind:

- **Independent schools have no GCSE results in the data.** So their shift cannot be split into "through the GCSE profile" and "beyond it", as it was for the state types. It contains whatever their GCSE profile would have been: a higher GCSE profile would run through the GCSE link at about 0.1 points per SD, so it would account for only part of a gap of 0.2 or more.
- **Only independent schools with published A-level value added are included** (576), and they are selective and fee-paying. Value added is measured against each pupil's own prior attainment, but who enters, who is admitted and who stays differ from the state sector.

In [ ]:
ind = d3[:, :, 0]                       # (draws, group): independent schools
rows = []
for k, tname in enumerate(state4):
    r = {"contrast": f"independent minus {tname}"}
    for j, gname in enumerate(group_names):
        diff = ind[:, j] - ta[:, j, k]
        lo, hi = np.percentile(diff, [5.5, 94.5])
        r[gname] = f"{diff.mean():+.2f} [{lo:+.2f}, {hi:+.2f}]"
    rows.append(r)
display(pd.DataFrame(rows).set_index("contrast"))

fig, ax = plt.subplots(figsize=(9, 5.2))
for k in range(4):
    diff = ind - ta[:, :, k]
    lo, med, hi = np.percentile(diff, [5.5, 50, 94.5], axis=0)
    off = (k - 1.5) * 0.17
    for j in range(n_groups):
        ax.plot([lo[j], hi[j]], [j + off, j + off], color=colors4[k], linewidth=2.2, label=f"independent minus {state4[k]}" if j == 0 else None)
        ax.plot(med[j], j + off, "o", color=colors4[k], markersize=4)
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_yticks(range(n_groups), group_names); ax.invert_yaxis()
ax.set_xlabel("A-level value added, independent schools minus the state type (points)")
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 1.13), ncol=2, frameon=False)
plt.tight_layout(); plt.show()

# the same, in one number per group: independent against the average state school, and against the two academy routes' gap
avg_gap = pd.DataFrame({"independent minus average state type": ind.mean(axis=0),
                        "converter minus sponsor-led (for scale)": (ta[:, :, 0] - ta[:, :, 1]).mean(axis=0)}, index=group_names)
display(avg_gap.round(3))

## Where the gap between two types comes from

The model-implied difference in average A-level value added between two types, among schools with GCSE results, split into: the part that runs **through the GCSE profile** (general quality, and the tilt), the part that is a **type shift beyond GCSE**, and the rest (**geography, time as an academy and the shared A-level quality**, which include where the two types happen to be located). The parts add up to the total. This answers, for each subject group, whether an academy route's A-level gap is largely what its GCSE intake profile would predict or something more.

In [ ]:
D = {k: draws(k, 5) for k in ["b", "c", "lam_a", "g", "h", "u", "psi", "xi", "slope_u", "type_a"]}
n_draw = len(D["b"])
psi_all = np.concatenate([D["psi"], np.zeros((n_draw, 1))], axis=1)
shared_all = psi_all[:, reg_idx] + D["xi"][:, la_idx] + (is_cs * years_c)[None, :] * D["slope_u"][:, cs_idx] + np.concatenate([D["u"][:, :n_g], D["u"][:, n_g:]], axis=1)

def gap(a_name, b_name):
    ia, ib = state4.index(a_name), state4.index(b_name)
    sa, sb = inst_type[:n_g] == type_names.index(a_name), inst_type[:n_g] == type_names.index(b_name)
    dg = D["g"][:, sa].mean(axis=1) - D["g"][:, sb].mean(axis=1)
    dh = D["h"][:, sa].mean(axis=1) - D["h"][:, sb].mean(axis=1)
    ds = shared_all[:, :n_g][:, sa].mean(axis=1) - shared_all[:, :n_g][:, sb].mean(axis=1)
    parts = {"through general GCSE quality": D["b"] * dg[:, None], "through the tilt": D["c"] * dh[:, None],
             "type shift beyond GCSE": D["type_a"][:, :, ia] - D["type_a"][:, :, ib],
             "geography, time as academy, shared quality": D["lam_a"] * ds[:, None]}
    parts["total"] = sum(parts.values())
    rows = []
    for j, gname in enumerate(group_names):
        r = {"group": gname}
        for k, v in parts.items():
            lo, hi = np.percentile(v[:, j], [5.5, 94.5])
            r[k] = f"{v[:, j].mean():+.3f} [{lo:+.3f}, {hi:+.3f}]"
        r["P(total > 0)"] = round(float((parts["total"][:, j] > 0).mean()), 2)
        rows.append(r)
    return pd.DataFrame(rows).set_index("group")

print("academy converter minus academy sponsor-led (value-added points):")
display(gap("academy converter", "academy sponsor-led"))
print("academy converter minus LA maintained:")
display(gap("academy converter", "LA maintained"))
print("academy sponsor-led minus LA maintained:")
display(gap("academy sponsor-led", "LA maintained"))
import gc; gc.collect()

## How much of the variation between schools does type explain?

The share of each subject group's true school-to-school variance in A-level value added, among schools with GCSE results, split into: general GCSE quality and tilt (which include the type and geography shifts in those quantities), the type shift **beyond** GCSE, the shared A-level quality (geography, time as an academy and the residual $u_i$), and the group's own scatter.

In [ ]:
D2 = {k: draws(k, 5) for k in ["b", "c", "lam_a", "g", "h", "sd_group", "type_a"]}
t4_g = t4_idx[:n_g]
type_beyond = D2["type_a"][:, :, :][:, :, t4_g] * is4[:n_g][None, None, :]          # (n, group, schools)
comp = {"general GCSE quality": D2["b"]**2 * D2["g"].var(axis=1)[:, None], "tilt": D2["c"]**2 * D2["h"].var(axis=1)[:, None],
        "type beyond GCSE": type_beyond.var(axis=2),
        "shared A-level quality": D2["lam_a"]**2 * shared_all[:, :n_g].var(axis=1)[:, None],
        "group's own scatter": D2["sd_group"]**2}
total = sum(comp.values())
tab = pd.DataFrame({k: (v / total).mean(axis=0) for k, v in comp.items()}, index=group_names)
tab["true SD"] = np.sqrt(total).mean(axis=0)
display(tab.round(3))
del D, D2, shared_all, type_beyond; gc.collect()

## Do the type gaps differ in London?

Everything above assumes a type gap is the same in every region. That is tested in a separate notebook, `a-level-institution-type-london.ipynb`, which refits the model with a London-specific extra shift for each state type (in general GCSE quality, and in A-level value added beyond GCSE for each subject group) and compares the gaps in London with those elsewhere. It is a separate notebook because the second fit does not fit in this notebook's memory alongside the first.

## Summary

Read with the framing at the top: one year of results, so association, and the two academy routes started from different places.

**Region and local authority (in the model throughout, so every type comparison is net of location)**

- **London is highest at GCSE** by a wide margin: $+0.71$ SD (about 0.32 GCSE value-added points) above the average region, then the South East ($+0.21$ SD, 0.09 points), with the North East lowest ($-0.39$ SD, $-0.17$ points).
- **Beyond the GCSE profile, regions barely differ at A-level:** the regional shift in A-level quality is within about 0.01 points either way in every region, so regional differences at A-level are carried almost entirely by GCSE quality and by the mix of types.
- **Boroughs differ a lot within London:** general GCSE quality ranges from about $+0.55$ SD (Barnet) to $-0.45$ SD (Greenwich) relative to London's average, so a borough matters as much as a region.
- **The type gaps are not all the same in London.** In `a-level-institution-type-london.ipynb` the converter versus sponsor-led gap at GCSE is about half as large in London ($+0.41$ SD against $+0.77$ elsewhere), while the A-level gaps beyond GCSE are not shown to differ.

**At GCSE, after region and local authority**

- **Converters sit clearly above sponsor-led academies** in general quality: $+0.71$ SD [0.60, 0.82], about 0.32 GCSE value-added points, and are more consistent across elements (log consistency $-0.12$ [-0.19, -0.06]). Their tilt does not differ.
- **Against LA-maintained schools,** converters are above ($+0.39$ SD, about 0.18 points) and sponsor-led academies below ($-0.32$ SD, about 0.14 points).
- **Free schools, UTCs and studio schools** are below maintained schools in general quality ($-0.29$ SD, about 0.13 points), lean strongly to the Maths-and-Science side ($+0.80$ [0.53, 1.06]) and are more scattered across elements ($+0.47$). That fits a technical focus, but this category mixes three kinds of school and we have not separated them.
- **Time as an academy:** longer-established converters and sponsor-led academies both sit higher in general quality, by about 0.38 and 0.31 SD per decade (0.17 and 0.14 points). For the shared A-level quality the slopes are small (about 0.03-0.04 points per decade) and their intervals include zero.

**At A-level, beyond the GCSE profile, by subject group** (value-added points)

- **Converters do better than sponsor-led academies in every group,** even after the GCSE profile is allowed for: Maths $+0.10$ [0.05, 0.16], Sciences $+0.11$, Humanities $+0.07$, Business & Computing $+0.14$ [0.09, 0.19], Creative arts $+0.11$, English $+0.05$ [0.01, 0.10], Social sciences $+0.04$ [0.00, 0.08].
- **Converters look like maintained schools** once GCSE is allowed for, except in the Sciences ($+0.06$ [0.02, 0.10]).
- **Sponsor-led academies sit below both.** Against maintained schools the shortfall is clear (interval excludes zero) in Maths ($-0.10$), Humanities ($-0.06$) and Business & Computing ($-0.13$), and negative but with intervals touching zero in Sciences, English, Social sciences and Creative arts.
- **Independent schools sit above every state type in every subject group,** with all intervals clear of zero (about 0.19 to 0.29 points above the average state type, Creative arts 0.45). Against converters the gap is +0.12 (Maths) to +0.26 (Humanities), Creative arts +0.42; against LA-maintained schools +0.12 to +0.27, Creative arts +0.44; against sponsor-led academies +0.22 to +0.35, Creative arts +0.53; against free schools/UTCs +0.14 (English) to +0.35 (Maths and Sciences), Creative arts +0.42. These gaps are larger than the converter versus sponsor-led gap (0.04 to 0.14), but they cannot be split into GCSE profile and beyond, because independent schools have no GCSE results, and only independent schools with published A-level value added are included (576), which are selective and fee-paying.
- **Colleges** are 0 to 0.13 points below the average state type (clearest in Humanities $-0.13$ and Business & Computing $-0.12$), again including whatever their unknown GCSE profile would have been.

**Where the converter versus sponsor-led gap comes from.** In total 0.10 (Social sciences) to 0.18 (Business & Computing) points, and about **one third runs through the GCSE profile** (0.04 to 0.06) and **two thirds is a type shift beyond it** (0.04 to 0.14). For Maths: 0.157 in total, 0.055 through GCSE and 0.104 beyond. Geography, time as an academy and the shared A-level quality make no difference to the gap. Against maintained schools, converters are +0.00 to +0.08 in total (only the Sciences clearly positive) and sponsor-led academies $-0.08$ to $-0.16$.

**Type explains little of the variation between schools.** The type shift beyond GCSE accounts for 0.6% to 3.8% of the true school-to-school variance in each group (largest in Sciences, Business & Computing and Maths), and most variation is within a type.

**What this can and cannot say about academy conversion**

- It is consistent with converters being stronger schools that were chosen to convert, and with sponsor-led academies being created where results were weak. The GCSE gap is what that history would predict.
- The A-level gap **beyond** the GCSE profile is larger than that history alone would explain, but it has other possible sources: who stays for or enters the sixth form, subject choice, and the fact that A-level students took their GCSEs two years earlier, at a school that may have been different then.
- Nothing here measures the effect of converting. That needs results for the same schools before and after conversion, over several years.

**Sampling.** Four chains of 500 draws: 3 divergences out of 2,000 and no chain-orientation problems. The weakest effective sample sizes (48 to 100) are for the spreads of the geographic effects and the overall GCSE level $\mu_e$; the type parameters in the diagnostics table have 210 to 280. The results above are stable to the run length (they moved by rounding only against a longer run).

**Caveats.** Free schools are grouped with UTCs and studio schools. Independent schools and colleges have no GCSE results, so their offsets cannot be separated from their GCSE profile. The "other" category is too small and mixed to interpret. Time as an academy uses the register's opening date.

**Next.** (a) Multi-year results, to look at schools before and after conversion. (b) The multi-academy trust, which the register records for about 60% of institutions, as a further level of the model. (c) Separating free schools, UTCs and studio schools. (d) Sixth-form size and entry policy, to test the intake explanation for the A-level gap.